# Multilingual RAG on the EU AI Act

A small RAG system that answers questions about the EU AI Act in English, French and Russian, and gives a link to the exact article it used. It follows the logic of the RAG practical session (TP) of my Master's course.

- **Documents:** the 113 articles of the AI Act (official English text), from the Hugging Face dataset `jeroenherczeg/eu-ai-act`
- **Questions:** 10 questions written in English and translated into French and Russian (30 questions in total), plus 5 questions with no answer in the AI Act
- **Main experiment:** does a multilingual embedding model (`bge-m3`) find the right article better than an English-only model (`gte-small`) when the question is not in English? Metric: Recall@5.

## 1. Installation

In [1]:
!pip install langchain-community langchain-huggingface langchain-text-splitters
!pip install datasets sentence-transformers faiss-cpu smolagents

## 2. Imports

In [2]:
import pandas as pd
from collections import Counter
from datasets import load_dataset
from transformers import AutoTokenizer

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

/tmp/ipykernel_4426/1430006645.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## 3. Data: the articles of the AI Act

We load the dataset and keep only the full articles in English (no recitals, no annexes).

In [3]:
ds = load_dataset("jeroenherczeg/eu-ai-act", split="train")

# Check
# What does the dataset contain?
print(Counter(ds["chunk_type"]))
print(Counter(ds["language"]))

# Keep only the full articles, in English
articles = ds.filter(lambda r: r["language"] == "en" and r["chunk_type"] == "article_full")
print(len(articles), "articles")  # expected: 113

Counter({'paragraph': 1566, 'recital': 540, 'article_full': 339, 'annex_item': 162})
Counter({'en': 869, 'nl': 869, 'fr': 869})
113 articles


### How long are the articles?

As in the TP, we look at the lengths to decide whether chunking is needed (1 token ≈ 4 characters). Some articles are much longer than others, so the long ones are cut into smaller chunks.

In [4]:
df = articles.to_pandas()          # turn the dataset into a pandas table
lengths = df["text"].str.len()     # number of characters in each article

print("Total:", lengths.sum(), "characters")
print("Shortest article:", lengths.min(), "characters")
print("Longest article:", lengths.max(), "characters")

Total: 290242 characters
Shortest article: 226 characters
Longest article: 17104 characters


### Documents with metadata

Each article becomes a `Document`: its text, plus metadata that we will use later:
- `source`: the name of the article, e.g. "Art. 5 AI Act"
- `article_no`: the article number, to evaluate the retrieval
- `url`: a link to the exact article on EUR-Lex

In [5]:
source_articles = []
for article in articles:
    number = article["article_no"]
    doc = Document(
        page_content=article["text"],
        metadata={
            "source": article["citation_label"],
            "article_no": number,
            "url": "https://eur-lex.europa.eu/eli/reg/2024/1689/oj/eng#art_" + str(number),
        },
    )
    source_articles.append(doc)

print(source_articles[4].metadata)  # check one example

{'source': 'Art. 5 AI Act', 'article_no': 5, 'url': 'https://eur-lex.europa.eu/eli/reg/2024/1689/oj/eng#art_5'}


### Chunking

Long articles are split into chunks of at most 512 tokens, with an overlap of 64 tokens so that a sentence cut in two is not lost. Short articles stay in one piece. We count tokens with the tokenizer of our embedding model, `bge-m3`.

In [6]:
embedding_model_name = "BAAI/bge-m3"
tokenizer = AutoTokenizer.from_pretrained(embedding_model_name)

text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,
    chunk_size=512,
    chunk_overlap=64,
)

chunks = text_splitter.split_documents(source_articles)
print(len(chunks), "chunks from", len(source_articles), "articles")

197 chunks from 113 articles


In [7]:
# show the chunks as a table (one row per chunk)
pd.DataFrame({**c.metadata, "text": c.page_content} for c in chunks)

,source,article_no,url,text
0,Art. 1 AI Act,1,https://eur-lex.europa.eu/eli/reg/2024/1689/oj...,Article 1 — Subject matter`\n\n1. The purpose ...
1,Art. 2 AI Act,2,https://eur-lex.europa.eu/eli/reg/2024/1689/oj...,Article 2 — Scope\n\n1. This Regulation applie...
2,Art. 2 AI Act,2,https://eur-lex.europa.eu/eli/reg/2024/1689/oj...,4. This Regulation applies neither to public a...
3,Art. 2 AI Act,2,https://eur-lex.europa.eu/eli/reg/2024/1689/oj...,12. This Regulation does not apply to AI syste...
4,Art. 3 AI Act,3,https://eur-lex.europa.eu/eli/reg/2024/1689/oj...,Article 3 — Definitions
...,...,...,...,...
192,Art. 111 AI Act,111,https://eur-lex.europa.eu/eli/reg/2024/1689/oj...,Article 111 — AI systems already placed on the...
193,Art. 112 AI Act,112,https://eur-lex.europa.eu/eli/reg/2024/1689/oj...,Article 112 — Evaluation and review\n\n1. The ...
194,Art. 112 AI Act,112,https://eur-lex.europa.eu/eli/reg/2024/1689/oj...,"5. By 2 August 2028, the Commission shall eval..."
195,Art. 112 AI Act,112,https://eur-lex.europa.eu/eli/reg/2024/1689/oj...,"10. The Commission shall, if necessary, submit..."


## 4. Vector index (FAISS)

Each chunk is turned into a vector (embedding) with `bge-m3`, a multilingual model. The vectors are stored in a FAISS index.

In [8]:
# normalize_embeddings=True gives all the vectors the same length,
# so the search compares only their direction (their meaning)
embedding_model = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    encode_kwargs={"normalize_embeddings": True},
)

# Turn every chunk into a vector and store them in FAISS
vectordb = FAISS.from_documents(chunks, embedding=embedding_model)

print(f"Index created : {vectordb.index.ntotal} vectors in the index.")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Index created : 197 vectors in the index.


In [9]:
# Quick check: Article 5 (prohibited practices) should come first

my_questions = [
    "Which AI practices are prohibited?",
    "Quelles pratiques d'IA sont interdites ?",
    "Какие практики ИИ запрещены?",
]

for question in my_questions:
    print("Question:", question)
    results = vectordb.similarity_search(question, k=5)
    for doc in results:
        print("   ", doc.metadata["source"])
    print()

Question: Which AI practices are prohibited?
    Art. 5 AI Act
    Art. 5 AI Act
    Art. 59 AI Act
    Art. 1 AI Act
    Art. 5 AI Act

Question: Quelles pratiques d'IA sont interdites ?
    Art. 5 AI Act
    Art. 5 AI Act
    Art. 59 AI Act
    Art. 5 AI Act
    Art. 2 AI Act

Question: Какие практики ИИ запрещены?
    Art. 5 AI Act
    Art. 5 AI Act
    Art. 5 AI Act
    Art. 59 AI Act
    Art. 50 AI Act



## 5. The LLM

We use `Qwen/Qwen2.5-3B-Instruct`, running locally on the Colab GPU, because the Hugging Face API is paid.

In [17]:
from transformers import pipeline

# Download the model and run it on Colab's GPU
llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-3B-Instruct",
    device_map="auto",   # put the model on the GPU
    dtype="auto",
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [18]:
# Test the LLM alone first, to check that the token works
messages = [{"role": "user", "content": "Hello, how are you?"}]
output = llm(messages, max_new_tokens=100)
print(output[0]["generated_text"][-1]["content"])

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Hello! I'm an artificial intelligence, so I don't have feelings or emotions in the way that humans do. However, I'm here and ready to help you with any questions or tasks you might have! How can I assist you today?


## 6. The RAG system

`ask` does the 3 steps of RAG:
1. **Retrieval**: find the 5 chunks most similar to the question
2. **Prompt**: put the chunks and the question into the prompt
3. **Generation**: the LLM answers using only these chunks

The links to EUR-Lex are printed by Python from the metadata, not written by the LLM, so they can't be invented.

Each question is answered independently (no conversation history, unlike the TP's loop): each answer depends only on its own question.

In [26]:
rag_prompt = """You are an assistant specialised in the EU AI Act.
Use ONLY the sources below to answer the question.
The question may use different words than the sources: if a source is about the same topic, use it to answer.
Answer in the same language as the question, in 3 to 5 sentences.
Mention the article you used, for example (Art. 50 AI Act).
Only if none of the sources is about the topic of the question, reply exactly "I don't know", in English.

SOURCES:
{context}

QUESTION: {query}

ANSWER:"""


def ask(query, k=5):
    # 1. Retrieval
    docs = vectordb.similarity_search(query, k=k)

    # 2. Prompt: each chunk with its article name in brackets
    context = "\n\n".join(f"[{d.metadata['source']}]\n{d.page_content}" for d in docs)
    prompt = rag_prompt.format(context=context, query=query)

    # 3. Generation
    output = llm([{"role": "user", "content": prompt}], max_new_tokens=400)
    answer = output[0]["generated_text"][-1]["content"]

    # 4. Print the query, the answer and the sources
    print("Query:", query)
    print("Answer:", answer)

    if "I don't know" not in answer: # no sources when there is no answer
        print("Sources:")
        printed = []
        for d in docs:
            if d.metadata["url"] not in printed: # don't print the same article twice
                print("  -", d.metadata["source"], d.metadata["url"])
                printed.append(d.metadata["url"])
    print()

In [27]:
# Demo: the same question in the 3 languages
ask("Which AI practices are prohibited?")
ask("Quelles pratiques d'IA sont interdites ?")
ask("Какие практики ИИ запрещены?")

[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Query: Which AI practices are prohibited?
Answer: According to Article 5 of the AI Act, several AI practices are prohibited:

(a) Placing on the market, putting into service, or using an AI system that employs subliminal techniques or purposefully manipulative or deceptive techniques to distort a person's behavior, leading to significant harm.
(b) Using an AI system that exploits vulnerabilities of individuals based on age, disability, or social/economic status, causing significant harm.
(c) Using AI systems to evaluate or classify natural persons or groups of persons over a period of time based on social behavior, creating or expanding facial recognition databases through untargeted scraping of facial images from the internet or CCTV footage, or inferring emotions in workplace and educational settings without medical or safety justifications.
(d) Using AI systems to create or expand biometric categorization systems that categorize natural persons based on biometric data to deduce or i

[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Query: Quelles pratiques d'IA sont interdites ?
Answer: Selon l'article 5 de la Loi sur l'IA européenne (AI Act), les pratiques d'IA interdites comprennent :

(a) L'implantation sur le marché, la mise en service ou l'utilisation d'un système d'IA qui déploie des techniques subliminales au-delà du niveau conscient de la conscience d'une personne ou des techniques manipulatrices ou trompeuses intentionnellement, avec l'objectif, ou l'effet, de déréguler significativement la conduite d'une personne ou d'une groupe de personnes en les rendant incapable de prendre une décision éclairée, ce qui entraîne ou est raisonnablement susceptible d'entraîner un dommage important pour cette personne, une autre personne ou un groupe de personnes ;

(b) L'implantation sur le marché, la mise en service ou l'utilisation d'un système d'IA qui exploite les vulnérabilités d'une personne naturelle ou d'un groupe spécifique à cause de leur âge, de leur handicap ou de leur situation sociale ou économique partic

## 7. Evaluation

The file `questions.csv` contains the test questions.

It has 4 columns:
- `id`: id of a question
- `lang`: the language of the question (en, fr or ru)
- `question`: the question
- `gold_article`: the number of the article that contains the answer. It is **0** for the questions whose answer is not in the AI Act.

We use 10 questions in English and translate them into French and Russian, plus 5 questions with no answer. Because the questions are the same in the 3 languages, only the language changes.

In [28]:
questions = pd.read_csv("questions.csv")
answerable = questions[questions["gold_article"] != 0]

print(len(questions), "questions in total")
print(len(answerable), "questions with an answer")
questions

35 questions in total
30 questions with an answer


,id,lang,question,gold_article
0,1,en,"If a company puts a chatbot on its website, do...",50
1,1,fr,Si une entreprise installe un chatbot sur son ...,50
2,1,ru,Если компания размещает чат-бота на своём сайт...,50
3,2,en,Can my employer use software that reads my fac...,5
4,2,fr,Mon employeur peut-il utiliser un logiciel qui...,5
5,2,ru,"Может ли работодатель использовать программу, ...",5
6,3,en,How much money can a company be forced to pay ...,99
7,3,fr,Combien une entreprise peut-elle devoir payer ...,99
8,3,ru,Какую сумму может быть вынуждена заплатить ком...,99
9,4,en,If a bank's AI system played a part in refusin...,86


### Test 1, Retrieval: Recall@5

For each question: is the right article among the 5 retrieved chunks? Recall@5 is the share of questions where the answer is yes (1.0 = always).

We compare two embedding models, with exactly the same chunks and the same questions, so the model is the only difference:
- `gte-small`: the TP's model, English only
- `bge-m3`: multilingual

In [29]:
def recall_at_5(db, qs):
    hits = []
    for _, row in qs.iterrows():
        docs = db.similarity_search(row["question"], k=5)
        found_articles = [d.metadata["article_no"] for d in docs]
        hits.append(row["gold_article"] in found_articles)  # True or False
    qs = qs.copy()
    qs["hit"] = hits
    return qs.groupby("lang")["hit"].mean()  # % of True, per language

In [30]:
# Second index, with the English-only model
emb_small = HuggingFaceEmbeddings(
    model_name="thenlper/gte-small",
    encode_kwargs={"normalize_embeddings": True},
)

vectordb_small = FAISS.from_documents(chunks, emb_small)

results = pd.DataFrame({
    "gte-small (EN only)": recall_at_5(vectordb_small, answerable),
    "bge-m3 (multilingual)": recall_at_5(vectordb, answerable),
})
results

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,gte-small (EN only),bge-m3 (multilingual)
lang,,
en,0.4,0.9
fr,0.3,0.9
ru,0.0,0.9


*My notes on the output:* the English-only model gets worse as the language gets further from English (EN → FR → RU), while bge-m3 stays at 0.9 in all three languages.

In [32]:
for i in range(len(answerable)):
    row = answerable.iloc[i]
    results = vectordb.similarity_search(row["question"], k=5)
    articles_found = []
    for doc in results:
        articles_found.append(doc.metadata["article_no"])
    if row["gold_article"] not in articles_found:
        print(row["lang"], "| expected:", row["gold_article"], "| found:", articles_found)
        print("   ", row["question"])

en | expected: 99 | found: [5, 101, 100, 5, 100]
    How much money can a company be forced to pay if it uses one of the banned AI techniques?
fr | expected: 2 | found: [100, 42, 47, 78, 59]
    Une armée doit-elle respecter les règles européennes sur l'IA lorsqu'elle développe une IA à des fins de défense ?
ru | expected: 2 | found: [75, 42, 47, 40, 100]
    Должна ли армия соблюдать европейские правила об ИИ, когда разрабатывает ИИ для целей обороны?


### Test 2, Generation: manual

- 5 answerable questions: are the answers correct, and do they cite the right article?
- The unanswerable questions: does the system say "I don't know"?

In [31]:
print("===== First 5 questions =====")
print()
for i in range(5):
    ask(questions["question"][i])

print("===== Questions with no answer =====")
print()
for i in range(len(questions)):
    if questions["gold_article"][i] == 0:
        ask(questions["question"][i])

[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


===== First 5 questions =====



[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Query: If a company puts a chatbot on its website, does it have to tell visitors that they are not talking to a real person?
Answer: According to Article 50 of the EU AI Act, if the chatbot interacts directly with natural persons, the company must inform them that they are interacting with an AI system. This obligation does not apply if the chatbot is authorized by law to detect, prevent, investigate, or prosecute criminal offences, provided there are appropriate safeguards for the rights and freedoms of third parties. Therefore, if the chatbot on the company's website is an AI system designed to interact directly with users, it must inform visitors of its artificial nature. Art. 50 AI Act.
Sources:
  - Art. 50 AI Act https://eur-lex.europa.eu/eli/reg/2024/1689/oj/eng#art_50
  - Art. 31 AI Act https://eur-lex.europa.eu/eli/reg/2024/1689/oj/eng#art_31
  - Art. 78 AI Act https://eur-lex.europa.eu/eli/reg/2024/1689/oj/eng#art_78
  - Art. 60 AI Act https://eur-lex.europa.eu/eli/reg/2024/16

[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Query: Can I claim copyright on an image that I generated with an AI tool?
Answer: I don't know



[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Query: Qui était le rapporteur du règlement sur l'IA au Parlement européen ?
Answer: I don't know

Les sources fournies ne contiennent aucune information concernant le rapporteur du règlement sur l'IA au Parlement européen. Les sources évoquent plutôt des détails sur le contenu et les aspects techniques du règlement sur l'IA (Artificial Intelligence Act), ainsi que les rôles des différentes autorités et organismes impliqués dans son application.



[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Query: Combien d'heures par jour un salarié a-t-il le droit d'utiliser des outils d'IA ?
Answer: I don't know

Les sources que j'ai consultées ne contiennent aucune information spécifique concernant le nombre d'heures par jour auxquelles un salarié a droit d'utiliser des outils d'IA. Elles traitent plutôt de la réglementation et des obligations relatives à l'utilisation d'outils d'IA dans le cadre professionnel, en particulier pour les systèmes d'IA de haut risque.

Query: Какие правила регулируют искусственный интеллект в США?
Answer: I don't know

The provided sources are exclusively about the EU's AI Act (Art. 57 AI Act, Art. 1 AI Act, etc.), and there is no information about the regulation of artificial intelligence in the USA.



*My notes on the output:* The answers are correct in English, while the French and Russian answers contain more errors (citations, terminology, grammar), which is a typical limit of a small 3B model.



## 8. Conclusion

**Results:**

| Recall@5 | gte-small (English only) | bge-m3 (multilingual) |
|---|---|---|
| en | 0.4 | 0.9 |
| fr | 0.3 | 0.9 |
| ru | 0.0 | 0.9 |

- The English-only model gets worse as the language of the question moves further from English: 4/10 in English, 3/10 in French, 0/10 in Russian.
- The multilingual model finds the right article for 9 questions out of 10 in all three languages: the language of the question makes no difference.
- gte-small is also weak in English (0.4) because the questions are paraphrased and do not reuse the words of the law. So bge-m3 is a better search model in general, and it does not lose anything when the language changes.
- bge-m3 misses 3 of the 30 questions. The fines question (EN) finds neighbouring articles about fines (100, 101) instead of 99. The army question (FR, RU) misses Article 2, a long article where the military exception is only one sentence among many other topics.
- Generation: the 5 checked answers give the correct rule, in the language of the question. The 5 questions with no answer all get "I don't know".

**Limits:**
- The search always retrieves 5 chunks. For a question about one article, some chunks are noise; for a question with no answer, 5 irrelevant chunks are still sent to the LLM. The 5 chunks can also contain the same article several times.
- Long articles are cut into chunks, and one sentence can be diluted among other topics in its chunk (example: the military exception in Article 2).
- Only the articles are indexed (no annexes, no recitals).
- Few questions (10 × 3 languages + 5): the results are only an indication.
- The LLM is a small free model (Qwen2.5-3B, run in Colab). Its French and Russian answers contain more errors than the English ones: a wrong article number (Art. 50 instead of Art. 5), the law called a "traité" or a "directive" instead of a "règlement", and some grammar mistakes in Russian.
- The prompt is a trade-off: with a strict prompt, the model answered "I don't know" to every question, even the answerable ones; with a more relaxed prompt, it answers, but could invent more.

**Possible improvements:**
- Use smaller chunks, or index the law paragraph by paragraph, so that one sentence is not diluted among other topics.
- Add a multilingual reranker to sort the retrieved chunks better (as in the Cohere notebook, recommended by TP).
- Use an agent that can do several searches for one question (as at the end of the TP).
- Use a bigger LLM for better answers in French and Russian.

## 9. Try it yourself

Run this cell to chat with the system. Type `exit` to stop.

In [ ]:
question = input("Your question: ")
while question != "exit":
    ask(question)
    question = input("Your question: ")